In [1]:
result_folder="" #place the results folder

In [2]:
import json
import os
dataset_folder="" #place dataset path

with open(os.path.join(dataset_folder,"test-real.json"), "r", newline="") as f:
    test=json.load(f)



In [3]:
#Convert Polarity Label

def convertPolarity(dataset):
  new_dataset=list()

  for entry in dataset:
    new_entry=dict()
    new_entry["doc_id"]=entry["doc_id"]
    new_entry["text"]=entry["text"]
    new_entry["annotations"]=list()

    for annotation in entry["annotations"]:
      new_annotation=annotation.copy()
      if annotation["label"]=="Alergias medicamentosas":
        if annotation["Polaridade"]=="Positiva":
            new_annotation["label"]="Alergias medicamentosas positiva"
        else:
            new_annotation["label"]="Alergias medicamentosas negativa"
      else:
        new_annotation["label"]=annotation["label"]
      new_entry["annotations"].append(new_annotation)
    new_dataset.append(new_entry)
  return new_dataset


In [4]:

test=convertPolarity(test)

In [5]:
test[0]

{'doc_id': 2,
 'text': 'afd27e61-6b20-43f9-a7cd-c5986d3ecfa8\r\n\r\nCENTRO HOSPITALAR DE LISBOA - SERVIÇO DE MEDICINA INTERNA\r\nDirector de Serviço: Prof Doutor Leonardo da Vinci\r\nEquipa assistente: Dra Sofia Rodrigues; Dr Bruno Fernandes; Dr Joao Paulo Silva; Dr Maria Helena Costa; Dr Pedro Miguel Santos\r\nDoente admitido durante a manhã de hoje para Medicina Interna com diagnóstico de Sepse.\r\n\r\nIDENTIFICAÇÃO\r\nDoente de 67 anos de idade; Engenheiro Civil reformado; vive sozinho\r\n\r\nMOTIVO DE INTERNAMENTO\r\nSepse de origem abdominal com descompensaçao de Insuficiência Cardíaca e insuficiência renal aguda.\r\n\r\nANTECEDENTES\r\n# Hipertensão arterial, seguida no médico de família\r\n# Insuficiência Cardíaca com fração de ejeção ventricular esquerda diminuída, seguida no cardiologista\r\n# Diabetes Mellitus tipo 2, seguido no diabetologista\r\n# Ex-fumador 2 maços/dia (40 UMA), parou de fumar em janeiro de 2010\r\n\r\nMEDICAÇÃO HABITUAL\r\n- Enalapril 20 mg\r\n- Metformina

In [44]:
model="gemini-2.5-flash-lite"

In [45]:
prediction_struct=list()
import os
for file in os.listdir(os.path.join(result_folder,model)):
  if file.endswith(".json"):
    with open(os.path.join(result_folder,model,file), "r", newline="") as f:
      prediction=json.load(f)
      prediction["id"]=file.split(".")[0]

    prediction_struct.append(prediction)

In [46]:
prediction_struct[9]

{'extractions': [{'extraction_class': 'Diagnóstico',
   'extraction_text': 'Hipertensão arterial',
   'char_interval': {'start_pos': 804, 'end_pos': 824},
   'alignment_status': 'match_exact',
   'extraction_index': 1,
   'group_index': 0,
   'description': None,
   'attributes': {}},
  {'extraction_class': 'Diagnóstico',
   'extraction_text': 'dislipidemia',
   'char_interval': {'start_pos': 826, 'end_pos': 838},
   'alignment_status': 'match_exact',
   'extraction_index': 2,
   'group_index': 1,
   'description': None,
   'attributes': {}},
  {'extraction_class': 'Diagnóstico',
   'extraction_text': 'Doença renal crónica estágio 2',
   'char_interval': {'start_pos': 868, 'end_pos': 898},
   'alignment_status': 'match_exact',
   'extraction_index': 3,
   'group_index': 2,
   'description': None,
   'attributes': {}},
  {'extraction_class': 'Diagnóstico',
   'extraction_text': 'Doença pulmonar obstrutiva crónica',
   'char_interval': {'start_pos': 903, 'end_pos': 937},
   'alignment_st

## Compute Metrics

In [47]:
def get_gt_spans(entry):
    spans = []
    for ann in entry["annotations"]:
        spans.append((ann["begin"], ann["end"], ann["label"]))
    return spans

In [48]:
get_gt_spans(test[3])

[(898, 920, 'Alergias medicamentosas negativa'),
 (956, 990, 'Diagnóstico'),
 (3037, 3083, 'Medicação Habitual'),
 (3100, 3150, 'Medicação Habitual'),
 (3197, 3218, 'Medicação Habitual'),
 (3222, 3244, 'Medicação Habitual')]

In [49]:
def get_pred_spans(prediction):
    """Extract (begin, end, label) tuples from predictions."""
    spans = []
    for ext in prediction["extractions"]:
        try:
          spans.append((
              ext["char_interval"]["start_pos"],
              ext["char_interval"]["end_pos"],
              ext["extraction_class"]
          ))
        except Exception as e:
          print(e)
          print(prediction["document_id"])
          span_text=ext["extraction_text"]
          #get begin end from the original text based on span text
          begin=prediction["text"].find(span_text)
          end=begin+len(span_text)
          spans.append((begin,end,ext["extraction_class"]))


    return spans


In [50]:
for entry in prediction_struct:
  print(entry["id"])
  print(get_pred_spans(entry))

9
[(667, 686, 'Medicação Habitual'), (688, 703, 'Medicação Habitual'), (705, 723, 'Medicação Habitual'), (725, 744, 'Medicação Habitual'), (746, 765, 'Medicação Habitual'), (788, 802, 'Alergias medicamentosas positiva'), (890, 909, 'Diagnóstico'), (2600, 2631, 'Diagnóstico'), (2715, 2744, 'Diagnóstico'), (2772, 2790, 'Diagnóstico'), (3130, 3144, 'Alergias medicamentosas positiva'), (2793, 2830, 'Diagnóstico')]
34
[(596, 611, 'Medicação Habitual'), (615, 634, 'Medicação Habitual'), (638, 660, 'Medicação Habitual'), (664, 714, 'Medicação Habitual'), (718, 733, 'Medicação Habitual'), (1754, 1765, 'Medicação Habitual'), (1768, 1780, 'Medicação Habitual'), (1626, 1646, 'Diagnóstico')]
2
[(905, 920, 'Medicação Habitual'), (924, 942, 'Medicação Habitual'), (946, 964, 'Medicação Habitual'), (968, 993, 'Medicação Habitual'), (997, 1015, 'Medicação Habitual'), (2224, 2233, 'Medicação Habitual'), (2387, 2398, 'Medicação Habitual'), (2734, 2778, 'Diagnóstico'), (2910, 2919, 'Medicação Habitual'), 

In [51]:
# Exact match (default above)
def exact_match(pred, gt):
    return pred[0] == gt[0] and pred[1] == gt[1] and pred[2] == gt[2]


In [52]:
def spans_match(pred, gt):
    if pred[2] != gt[2]: #if label is different
        return False
    overlap_start = max(pred[0], gt[0])
    overlap_end   = min(pred[1], gt[1])
    overlap_len   = max(0, overlap_end - overlap_start)
    union_len     = (pred[1] - pred[0]) + (gt[1] - gt[0]) - overlap_len
    iou = overlap_len / union_len if union_len > 0 else 0
    return iou >= 0.5

In [53]:
from collections import defaultdict

def evaluate_relaxed(gt_entries, predictions):
    """
    gt_entries: list of ground truth dicts (your dataset format)
    predictions: list of prediction dicts (your extractions format), same order
    """


    # Per-class counters
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)


    for gt_entry, pred in zip(gt_entries, predictions):
        gt_spans = get_gt_spans(gt_entry)
        pred_spans = get_pred_spans(pred)

        matched_gt = set()
        matched_pred = set()

        for i, p in enumerate(pred_spans):
            for j, g in enumerate(gt_spans):
                if j not in matched_gt and spans_match(p, g):
                    tp[p[2]] += 1
                    matched_gt.add(j)
                    matched_pred.add(i)
                    break

        for i, p in enumerate(pred_spans):
            if i not in matched_pred:
                fp[p[2]] += 1

        for j, g in enumerate(gt_spans):
            if j not in matched_gt:
                fn[g[2]] += 1

    # Per-class metrics
    all_classes = set(list(tp.keys()) + list(fp.keys()) + list(fn.keys()))
    print(f"{'Class':<45} {'P':>6} {'R':>6} {'F1':>6} {'TP':>5} {'FP':>5} {'FN':>5}")
    print("-" * 80)

    total_tp = total_fp = total_fn = 0
    for cls in sorted(all_classes):
        t, f_p, f_n = tp[cls], fp[cls], fn[cls]
        total_tp += t; total_fp += f_p; total_fn += f_n
        p  = t / (t + f_p) if (t + f_p) > 0 else 0
        r  = t / (t + f_n) if (t + f_n) > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        print(f"{cls:<45} {p:>6.3f} {r:>6.3f} {f1:>6.3f} {t:>5} {f_p:>5} {f_n:>5}")

    # Micro F1
    print("-" * 80)
    micro_p  = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_r  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
    print(f"{'MICRO':<45} {micro_p:>6.3f} {micro_r:>6.3f} {micro_f1:>6.3f} {total_tp:>5} {total_fp:>5} {total_fn:>5}")
    # Macro F1
    class_f1s = []
    class_ps = []
    class_rs = []
    res_classes=list()
    for cls in all_classes:
        t, f_p, f_n = tp[cls], fp[cls], fn[cls]
        p  = t / (t + f_p) if (t + f_p) > 0 else 0
        r  = t / (t + f_n) if (t + f_n) > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        class_f1s.append(f1)
        class_ps.append(p)
        class_rs.append(r)
        res_classes.append({"CLASS":cls,"P":p,"R":r,"F1":f1})

    macro_p  = sum(class_ps)  / len(class_ps)  if class_ps  else 0
    macro_r  = sum(class_rs)  / len(class_rs)  if class_rs  else 0
    macro_f1 = sum(class_f1s) / len(class_f1s) if class_f1s else 0
    print(f"{'MACRO':<45} {macro_p:>6.3f} {macro_r:>6.3f} {macro_f1:>6.3f}")

    return {
        "micro_p": micro_p, "micro_r": micro_r, "micro_f1": micro_f1,
        "macro_p": macro_p, "macro_r": macro_r, "macro_f1": macro_f1,
    },res_classes


In [54]:
from collections import defaultdict

def evaluate_exact(gt_entries, predictions):
    """
    gt_entries: list of ground truth dicts (your dataset format)
    predictions: list of prediction dicts (your extractions format), same order
    """


    # Per-class counters
    tp = defaultdict(int)
    fp = defaultdict(int)
    fn = defaultdict(int)


    for gt_entry, pred in zip(gt_entries, predictions):
        gt_spans = get_gt_spans(gt_entry)
        pred_spans = get_pred_spans(pred)

        matched_gt = set()
        matched_pred = set()

        for i, p in enumerate(pred_spans):
            for j, g in enumerate(gt_spans):
                if j not in matched_gt and exact_match(p, g):
                    tp[p[2]] += 1
                    matched_gt.add(j)
                    matched_pred.add(i)
                    break

        for i, p in enumerate(pred_spans):
            if i not in matched_pred:
                fp[p[2]] += 1

        for j, g in enumerate(gt_spans):
            if j not in matched_gt:
                fn[g[2]] += 1

    # Per-class metrics
    all_classes = set(list(tp.keys()) + list(fp.keys()) + list(fn.keys()))
    print(f"{'Class':<45} {'P':>6} {'R':>6} {'F1':>6} {'TP':>5} {'FP':>5} {'FN':>5}")
    print("-" * 80)

    total_tp = total_fp = total_fn = 0
    for cls in sorted(all_classes):
        t, f_p, f_n = tp[cls], fp[cls], fn[cls]
        total_tp += t; total_fp += f_p; total_fn += f_n
        p  = t / (t + f_p) if (t + f_p) > 0 else 0
        r  = t / (t + f_n) if (t + f_n) > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        print(f"{cls:<45} {p:>6.3f} {r:>6.3f} {f1:>6.3f} {t:>5} {f_p:>5} {f_n:>5}")

    # Micro F1
    print("-" * 80)
    micro_p  = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    micro_r  = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    micro_f1 = 2 * micro_p * micro_r / (micro_p + micro_r) if (micro_p + micro_r) > 0 else 0
    print(f"{'MICRO':<45} {micro_p:>6.3f} {micro_r:>6.3f} {micro_f1:>6.3f} {total_tp:>5} {total_fp:>5} {total_fn:>5}")
    # Macro F1
    class_f1s = []
    class_ps = []
    class_rs = []
    res_classes=list()
    for cls in all_classes:
        t, f_p, f_n = tp[cls], fp[cls], fn[cls]
        p  = t / (t + f_p) if (t + f_p) > 0 else 0
        r  = t / (t + f_n) if (t + f_n) > 0 else 0
        f1 = 2 * p * r / (p + r) if (p + r) > 0 else 0
        class_f1s.append(f1)
        class_ps.append(p)
        class_rs.append(r)
        res_classes.append({"CLASS":cls,"P":p,"R":r,"F1":f1})

    macro_p  = sum(class_ps)  / len(class_ps)  if class_ps  else 0
    macro_r  = sum(class_rs)  / len(class_rs)  if class_rs  else 0
    macro_f1 = sum(class_f1s) / len(class_f1s) if class_f1s else 0
    print(f"{'MACRO':<45} {macro_p:>6.3f} {macro_r:>6.3f} {macro_f1:>6.3f}")

    return {
        "micro_p": micro_p, "micro_r": micro_r, "micro_f1": micro_f1,
        "macro_p": macro_p, "macro_r": macro_r, "macro_f1": macro_f1,
    } ,res_classes


In [55]:
test_entry=get_gt_spans(test[0])
pred_entry=get_pred_spans(prediction_struct[0])



In [56]:
print("EXACT_MATCH")
ov,res_class=evaluate_exact(test,prediction_struct)




EXACT_MATCH
'NoneType' object is not subscriptable
doc_040bebdb
Class                                              P      R     F1    TP    FP    FN
--------------------------------------------------------------------------------
Alergias medicamentosas negativa               0.000  0.000  0.000     0     2     4
Alergias medicamentosas positiva               0.667  0.750  0.706     6     3     2
Diagnóstico                                    0.045  0.333  0.080     3    63     6
Medicação Habitual                             0.560  0.723  0.631    47    37    18
--------------------------------------------------------------------------------
MICRO                                          0.348  0.651  0.453    56   105    30
MACRO                                          0.318  0.452  0.354


In [57]:
import pandas as pd
res_class_df=pd.DataFrame.from_dict(res_class)
res_class_df

,CLASS,P,R,F1
0,Alergias medicamentosas negativa,0.000000,0.000000,0.000000
1,Diagnóstico,0.045455,0.333333,0.080000
2,Medicação Habitual,0.559524,0.723077,0.630872
3,Alergias medicamentosas positiva,0.666667,0.750000,0.705882


In [58]:
#convert ov to pandas
ov_df=pd.DataFrame.from_dict(ov,orient="index")
ov_df.T

,micro_p,micro_r,micro_f1,macro_p,macro_r,macro_f1
0,0.347826,0.651163,0.453441,0.317911,0.451603,0.354189


In [59]:
print("RELAX_MATCH")
ov_relax,res_class_relax=evaluate_relaxed(test,prediction_struct)

RELAX_MATCH
'NoneType' object is not subscriptable
doc_040bebdb
Class                                              P      R     F1    TP    FP    FN
--------------------------------------------------------------------------------
Alergias medicamentosas negativa               0.500  0.250  0.333     1     1     3
Alergias medicamentosas positiva               0.667  0.750  0.706     6     3     2
Diagnóstico                                    0.045  0.333  0.080     3    63     6
Medicação Habitual                             0.560  0.723  0.631    47    37    18
--------------------------------------------------------------------------------
MICRO                                          0.354  0.663  0.462    57   104    29
MACRO                                          0.443  0.514  0.438


In [60]:
res_class_df_relax=pd.DataFrame.from_dict(res_class_relax)
res_class_df_relax

,CLASS,P,R,F1
0,Alergias medicamentosas negativa,0.500000,0.250000,0.333333
1,Diagnóstico,0.045455,0.333333,0.080000
2,Medicação Habitual,0.559524,0.723077,0.630872
3,Alergias medicamentosas positiva,0.666667,0.750000,0.705882


In [61]:
ov_df_relax=pd.DataFrame.from_dict(ov_relax,orient="index")
ov_df_relax.T

,micro_p,micro_r,micro_f1,macro_p,macro_r,macro_f1
0,0.354037,0.662791,0.461538,0.442911,0.514103,0.437522
